In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/"
gold_adls_path = "abfss://gold@"+storage_account+".dfs.core.windows.net/"

In [0]:
claims = spark.read.format("delta").load(silver_adls_path+"/claims")
enrollment = spark.read.format("delta").load(silver_adls_path+"/customer_enrollment")
policies = spark.read.format("delta").load(silver_adls_path+"/policy_products")
hospitals = spark.read.format("delta").load(silver_adls_path+"/hospitals")

# hospitals.printSchema()

root
 |-- hospital_id: long (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- hospital_type: string (nullable = true)
 |-- hospital_tier: string (nullable = true)
 |-- network_flag: boolean (nullable = true)
 |-- network_category: string (nullable = true)
 |-- empanelment_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- bed_capacity: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- average_claim_cost: double (nullable = true)
 |-- high_cost_hospital_flag: integer (nullable = true)



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

fact_claims = (
    claims.alias("c").join(hospitals.alias("h"), col("c.hospital_id") == col("h.hospital_id"), "left")
)

fact_claims = (
    fact_claims
    .select(
        col("c.claim_id"),
        col("c.claim_number"),
        col("c.claim_date"),
        col("c.claim_status"),

        col("c.total_bill_amount"),
        col("c.claimed_amount"),
        col("c.approved_amount"),
        col("c.rejected_amount"),

        col("c.length_of_stay"),
        col("c.approval_rate"),
        col("c.rejection_rate"),

        col("c.processing_days"),

        col("c.claim_year"),
        col("c.claim_month"),

        # hospital attributes
        col("h.hospital_name"),
        col("h.hospital_type"),
        col("h.network_flag"),
        col("h.city"),
        col("h.state"),
        col("h.region"),
        col("h.rating"),
        col("h.high_cost_hospital_flag")
    )
)

window = Window.partitionBy(col("claim_id")).orderBy(col("claim_date").desc())

fact_claims = (
    fact_claims
    .withColumn("rnk", rank().over(window))
    .filter(col("rnk") == 1)
    .drop("rnk")
)

fact_claims.write \
.format("delta") \
.mode("overwrite") \
.partitionBy("claim_year") \
.save(gold_adls_path+"/fact_claims")